In [53]:
# Cell 1 — Install
# Run once, then comment out
%pip install rioxarray richdem pysheds rasterstats geopandas requests tqdm



Note: you may need to restart the kernel to use updated packages.


In [54]:
# Cell 2 — Imports
import os, requests, zipfile, io
import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray
import xarray as xr
from pathlib import Path
from tqdm import tqdm

RAW = Path("../data/raw")
PROC = Path("../data/processed")
GEO  = RAW / "geo"
GEO.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)

In [55]:
# Cell 3 — Download India district shapefile (GADM level-2, ~5 MB)
# Direct download — no login needed
gadm_url = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_IND_shp.zip"
gadm_zip = GEO / "gadm41_IND_shp.zip"

if not gadm_zip.exists():
    print("Downloading GADM India shapefile...")
    r = requests.get(gadm_url, stream=True)
    with open(gadm_zip, "wb") as f:
        for chunk in r.iter_content(1024*1024):
            f.write(chunk)
    with zipfile.ZipFile(gadm_zip) as z:
        z.extractall(GEO / "gadm")
    print("Done")

districts = gpd.read_file(GEO / "gadm" / "gadm41_IND_2.shp")
print(districts.shape, districts.columns.tolist())
districts.to_file(GEO / "india_districts.gpkg", driver="GPKG")

(676, 14) ['GID_2', 'GID_0', 'COUNTRY', 'GID_1', 'NAME_1', 'NL_NAME_1', 'NAME_2', 'VARNAME_2', 'NL_NAME_2', 'TYPE_2', 'ENGTYPE_2', 'CC_2', 'HASC_2', 'geometry']


In [56]:
# Cell 4 — Download SRTM DEM via elevation package (requires gdal)
# Alternative: direct download from NASA (no login for SRTM30)
# We use 30-arc-second (~1km) GMTED2010 from USGS — direct, no auth
# Bounding box: India approx 68-98E, 8-38N

gmted_url = "https://edcintl.cr.usgs.gov/downloads/sciweb1/shared/topo/downloads/GMTED/Grid_ZipFiles/mn30_grd.zip"
gmted_zip = GEO / "gmted_mn30.zip"

if not (GEO / "gmted").exists():
    print("Downloading GMTED2010 DEM (~200 MB)...")
    r = requests.get(gmted_url, stream=True, timeout=120)
    with open(gmted_zip, "wb") as f:
        for chunk in r.iter_content(1024*1024):
            f.write(chunk)
    with zipfile.ZipFile(gmted_zip) as z:
        z.extractall(GEO / "gmted")
    print("Done")
else:
    print("GMTED already downloaded")

GMTED already downloaded


In [57]:
%pip install elevation


Note: you may need to restart the kernel to use updated packages.


In [58]:
# Cell 5 — Get DEM via elevation package (SRTM, no zip issues)
import subprocess, os
import rioxarray as rxr

srtm_out = str(GEO / "india_dem_1km.tif")

if not os.path.exists(srtm_out):
    subprocess.run(
        ["eio", "clip", "--bounds", "68 8 98 38", "--output", srtm_out],
        check=True
    )

india_dem = rxr.open_rasterio(srtm_out, masked=True).squeeze()
india_dem = india_dem.where(india_dem > -500)
print("DEM saved:", india_dem.shape)

DEM saved: (3601, 3601)


In [59]:
# Cell 5 — Download pre-clipped India DEM (small, no memory crash)
import requests, os
import rioxarray as rxr

srtm_out = GEO / "india_dem_1km.tif"

if not srtm_out.exists():
    # GEBCO 2023 via direct URL — India bbox, already small
    url = "https://www.gebco.net/data_and_products/gridded_bathymetry_data/gebco_2023/gebco_2023_sub_ice_topo/GEBCO_2023_sub_ice_topo_n38.0_s8.0_w68.0_e98.0.tif"
    print("Downloading...")
    r = requests.get(url, timeout=120)
    with open(srtm_out, "wb") as f:
        f.write(r.content)

india_dem = rxr.open_rasterio(srtm_out, masked=True).squeeze()
india_dem = india_dem.where(india_dem > -500)
print("DEM shape:", india_dem.shape)

DEM shape: (3601, 3601)


In [60]:
# Cell 6 — Download ESA WorldCover 2021 tiles for India (10m, free)
# Tile index for India: several tiles. We'll use the 100m degraded mosaic
# from Copernicus Global Land — direct download, no login

# 300m Land Cover from ESA CCI — simpler download
lc_url = "https://dap.ceda.ac.uk/neodc/esacci/land_cover/data/land_cover_maps/v2.0.7/ESACCI-LC-L4-LCCS-Map-300m-P1Y-2015-v2.0.7.tif"

# Fallback: use MODIS MCD12C1 land cover from NASA open data portal
# This URL is a direct HTTP link (no Earthdata login for this version)
modis_lc_url = "https://ladsweb.modaps.eosdis.nasa.gov/archive/allData/6/MCD12C1/2020/001/MCD12C1.A2020001.006.2021358015525.hdf"
# NOTE: MODIS requires Earthdata login — use the ESA CCI 300m instead

print("For land cover, manually download from:")
print("https://maps.elie.ucl.ac.be/CCI/viewer/download.php")
print("Or use the GEE export approach in Cell 7")
print()
print("Saving a placeholder — script will use IMD-only SCRI if land cover missing")
lc_available = (GEO / "india_landcover.tif").exists()
print("Land cover available:", lc_available)

For land cover, manually download from:
https://maps.elie.ucl.ac.be/CCI/viewer/download.php
Or use the GEE export approach in Cell 7

Saving a placeholder — script will use IMD-only SCRI if land cover missing
Land cover available: False


In [61]:
# Cell 7 — Download SoilGrids clay content via WCS (no login needed)
# 250m resolution, top 5cm layer
# Direct OGC WCS endpoint — paste URL into browser to verify

def download_soilgrids(variable, depth, outfile):
    """
    variable: 'clay', 'soc' (soil organic carbon)
    depth: '0-5cm'
    """
    # SoilGrids REST API (no auth needed)
    bbox = "68.0,8.0,98.0,38.0"  # India
    url = (
        f"https://maps.isric.org/mapserv?map=/map/{variable}.map"
        f"&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage"
        f"&COVERAGEID={variable}_{depth}_mean"
        f"&FORMAT=image/tiff"
        f"&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326"
        f"&SUBSET=long({bbox.split(',')[0]},{bbox.split(',')[2]})"
        f"&SUBSET=lat({bbox.split(',')[1]},{bbox.split(',')[3]})"
        f"&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326"
    )
    if not outfile.exists():
        print(f"Downloading {variable} {depth}...")
        r = requests.get(url, timeout=180)
        with open(outfile, "wb") as f:
            f.write(r.content)
        print(f"Saved {outfile} ({outfile.stat().st_size/1e6:.1f} MB)")
    else:
        print(f"{outfile.name} already exists")

download_soilgrids("clay", "0-5cm", GEO / "india_clay_0_5cm.tif")
download_soilgrids("soc",  "0-5cm", GEO / "india_soc_0_5cm.tif")

india_clay_0_5cm.tif already exists
india_soc_0_5cm.tif already exists


In [ ]:
# Cell 8 — Download JRC Global Surface Water occurrence band
# Tile-based system. For India we need tiles: 70E_10N, 70E_20N, 70E_30N,
# 80E_10N, 80E_20N, 80E_30N, 90E_10N, 90E_20N
# Direct HTTP from JRC (no login)

jrc_tiles = [
    "occurrence_70E_10Nv1_4_2021.tif",
    "occurrence_70E_20Nv1_4_2021.tif",
    "occurrence_70E_30Nv1_4_2021.tif",
    "occurrence_80E_10Nv1_4_2021.tif",
    "occurrence_80E_20Nv1_4_2021.tif",
    "occurrence_80E_30Nv1_4_2021.tif",
    "occurrence_90E_10Nv1_4_2021.tif",
    "occurrence_90E_20Nv1_4_2021.tif",
]

jrc_base = "https://storage.googleapis.com/global-surface-water/downloads2021/occurrence/"
jrc_dir = GEO / "jrc_water"
jrc_dir.mkdir(exist_ok=True)

for tile in jrc_tiles:
    out = jrc_dir / tile
    if not out.exists():
        url = jrc_base + tile
        print(f"Downloading {tile}...")
        r = requests.get(url, timeout=120)
        if r.status_code == 200:
            with open(out, "wb") as f:
                f.write(r.content)
        else:
            print(f"  WARNING: {r.status_code} for {tile}")
    else:
        print(f"{tile} already exists")
print("JRC tiles done")

occurrence_70E_10Nv1_4_2021.tif already exists
occurrence_70E_20Nv1_4_2021.tif already exists
occurrence_70E_30Nv1_4_2021.tif already exists
occurrence_80E_10Nv1_4_2021.tif already exists
occurrence_80E_20Nv1_4_2021.tif already exists
occurrence_80E_30Nv1_4_2021.tif already exists
occurrence_90E_10Nv1_4_2021.tif already exists
occurrence_90E_20Nv1_4_2021.tif already exists
JRC tiles done


: 

In [ ]:
# Cell 9 — Merge JRC tiles and clip to India
import rioxarray as rxr
from rioxarray.merge import merge_arrays

jrc_files = sorted((GEO / "jrc_water").glob("*.tif"))
tiles = [rxr.open_rasterio(f, masked=True).squeeze() for f in jrc_files]

# Merge
water_merged = merge_arrays(tiles)
water_india = water_merged.rio.clip_box(minx=68, miny=8, maxx=98, maxy=38)

# Occurrence > 50% = permanent/semi-permanent water body
water_mask = (water_india > 50).astype("float32")
water_mask.rio.to_raster(GEO / "india_water_mask.tif")
print("Water mask saved:", water_mask.shape)

In [ ]:
import pandas as pd
import geopandas as gpd
import rasterio
import numpy as np

from rasterio.sample import sample_gen

In [ ]:
rf = pd.read_csv(
    "../data/processed/feature_matrix.csv"
)

print(rf.shape)

rf.head()

(4964, 7)


,lat,lon,mean_annual_rf,std_annual,cv,dry_days,heavy_days
0,8.25,77.00,1368.70980,409.68338,0.299321,253.051282,0.487179
1,8.25,77.25,1163.10000,317.94876,0.273363,265.871795,0.487179
2,8.25,77.50,749.88873,343.41086,0.457949,313.076923,0.282051
3,8.25,77.75,758.49920,328.97153,0.433714,294.102564,0.256410
4,8.50,76.75,1833.17690,363.57733,0.198332,233.205128,0.820513


In [ ]:
gdf = gpd.GeoDataFrame(
    rf,
    geometry=gpd.points_from_xy(
        rf.lon,
        rf.lat
    ),
    crs="EPSG:4326"
)

In [ ]:
with rasterio.open(
    "../data/raw/geospatial/dem_india.tif"
) as src:

    coords = [
        (x,y)
        for x,y in zip(
            rf.lon,
            rf.lat
        )
    ]

    rf["elevation"] = [
        val[0]
        for val in src.sample(coords)
    ]

RasterioIOError: ../data/raw/geospatial/dem_india.tif: No such file or directory

In [ ]:
from pathlib import Path

for p in Path("../data").rglob("*"):
    print(p)

In [ ]:
import rasterio

dem = rasterio.open(
    "../data/raw/geo/india_dem_1km.tif"
)

print(dem)
print(dem.bounds)
print(dem.crs)

<open DatasetReader name='../data/raw/geo/india_dem_1km.tif' mode='r'>
BoundingBox(left=67.9998601191114, bottom=7.999860719111233, right=98.00819333241148, top=38.008193932411274)
EPSG:4326


In [ ]:
import pandas as pd
import rasterio

rf = pd.read_csv(
    "../data/processed/feature_matrix.csv"
)

coords = list(
    zip(
        rf.lon,
        rf.lat
    )
)

with rasterio.open(
    "../data/raw/geo/india_dem_1km.tif"
) as src:

    rf["elevation"] = [
        x[0]
        for x in src.sample(coords)
    ]

rf[
    ["lat","lon","elevation"]
].head()

,lat,lon,elevation
0,8.25,77.00,0.0
1,8.25,77.25,80.0
2,8.25,77.50,71.0
3,8.25,77.75,24.0
4,8.50,76.75,0.0


In [ ]:
rf["elevation"].describe()

count    4964.000000
mean      792.023193
std      1338.853394
min        -2.000000
25%       135.750000
50%       304.000000
75%       598.000000
max      6365.000000
Name: elevation, dtype: float64

In [ ]:
with rasterio.open(
    "../data/raw/geo/india_clay_0_5cm.tif"
) as src:

    rf["soil_clay"] = [
        x[0]
        for x in src.sample(coords)
    ]

rf["soil_clay"].describe()

count    4964.000000
mean      281.975423
std       100.451865
min         0.000000
25%       236.000000
50%       284.000000
75%       345.000000
max       529.000000
Name: soil_clay, dtype: float64

In [ ]:
import rasterio

rasterio.open(
    "../data/raw/geo/india_soc_0_5cm.tif"
)

<open DatasetReader name='../data/raw/geo/india_soc_0_5cm.tif' mode='r'>

In [ ]:
from pathlib import Path

p = Path("../data/raw/geo/india_soc_0_5cm.tif")

print("Exists:", p.exists())
print("Size MB:", p.stat().st_size/1024/1024)

Exists: True
Size MB: 101.29202651977539


In [ ]:
import rasterio

with rasterio.open(
    "../data/raw/geo/india_soc_0_5cm.tif"
) as src:

    print(src.width)
    print(src.height)
    print(src.count)
    print(src.crs)

13271
12553
1
EPSG:4326


In [ ]:
import rasterio

with rasterio.open(
    "../data/raw/geo/india_dem_1km.tif"
) as src:

    print(src.width)
    print(src.height)

3601
3601


In [ ]:
from pathlib import Path

for f in [
    "../data/raw/geo/india_dem_1km.tif",
    "../data/raw/geo/india_clay_0_5cm.tif",
    "../data/raw/geo/india_soc_0_5cm.tif"
]:
    p = Path(f)
    print(f)
    print("exists:", p.exists())
    print("size MB:", round(p.stat().st_size/1024/1024,2))
    print()


import rasterio

with rasterio.open("../data/raw/geo/india_dem_1km.tif") as src:
    print(src)

../data/raw/geo/india_dem_1km.tif
exists: True
size MB: 49.49

../data/raw/geo/india_clay_0_5cm.tif
exists: True
size MB: 88.62

../data/raw/geo/india_soc_0_5cm.tif
exists: True
size MB: 101.29

<open DatasetReader name='../data/raw/geo/india_dem_1km.tif' mode='r'>


In [ ]:
import rasterio

with rasterio.open("../data/raw/geo/india_dem_1km.tif") as src:
    print(src.bounds)

BoundingBox(left=67.9998601191114, bottom=7.999860719111233, right=98.00819333241148, top=38.008193932411274)


In [ ]:
import pandas as pd

rf = pd.read_csv("../data/processed/feature_matrix.csv")

print(rf[["lat","lon"]].head())

print("\nLatitude range:")
print(rf.lat.min(), rf.lat.max())

print("\nLongitude range:")
print(rf.lon.min(), rf.lon.max())

    lat    lon
0  8.25  77.00
1  8.25  77.25
2  8.25  77.50
3  8.25  77.75
4  8.50  76.75

Latitude range:
8.25 37.25

Longitude range:
68.0 97.25


In [ ]:
def safe_sample(src, x, y):
    try:
        return next(src.sample([(x, y)]))[0]
    except:
        return np.nan

In [ ]:
with rasterio.open("../data/raw/geo/india_dem_1km.tif") as src:

    rf["elevation"] = [
        safe_sample(src, lon, lat)
        for lon, lat in zip(rf.lon, rf.lat)
    ]

In [ ]:
import rasterio
import pandas as pd

rf = pd.read_csv("../data/processed/feature_matrix.csv")

with rasterio.open("../data/raw/geo/india_dem_1km.tif") as src:

    point = [(77.0, 8.25)]

    print(list(src.sample(point)))

[array([0.], dtype=float32)]


In [ ]:
import rasterio
import numpy as np

with rasterio.open("../data/raw/geo/india_dem_1km.tif") as src:

    arr = src.read(1)

print("Min:", np.nanmin(arr))
print("Max:", np.nanmax(arr))
print("Mean:", np.nanmean(arr))


with rasterio.open("../data/raw/geo/india_dem_1km.tif") as src:
    print("NoData:", src.nodata)

Min: -45.0
Max: 8625.0
Mean: 1304.9573
NoData: None


In [ ]:
import rasterio

points = [
    (77.0, 8.25),   # South India
    (77.0, 28.5),   # North India
    (78.0, 15.0),   # Central India
    (76.0, 12.0),
]

with rasterio.open("../data/raw/geo/india_dem_1km.tif") as src:

    vals = list(src.sample(points))

print(vals)

[array([0.], dtype=float32), array([215.], dtype=float32), array([257.], dtype=float32), array([838.], dtype=float32)]


In [ ]:
import pandas as pd
import rasterio
import numpy as np
from tqdm import tqdm

rf = pd.read_csv(
    "../data/processed/feature_matrix.csv"
)

def safe_sample(src, lon, lat):

    try:
        return next(
            src.sample([(lon, lat)])
        )[0]

    except:
        return np.nan

In [ ]:
with rasterio.open(
    "../data/raw/geo/india_dem_1km.tif"
) as src:

    rf["elevation"] = [

        safe_sample(src, lon, lat)

        for lon, lat in tqdm(
            zip(rf.lon, rf.lat),
            total=len(rf)
        )
    ]

100%|██████████| 4964/4964 [00:00<00:00, 12367.98it/s]


In [ ]:
rf["elevation"].describe()

count    4964.000000
mean      792.023193
std      1338.853394
min        -2.000000
25%       135.750000
50%       304.000000
75%       598.000000
max      6365.000000
Name: elevation, dtype: float64

In [ ]:
with rasterio.open(
    "../data/raw/geo/india_clay_0_5cm.tif"
) as src:

    rf["soil_clay"] = [

        safe_sample(src, lon, lat)

        for lon, lat in tqdm(
            zip(rf.lon, rf.lat),
            total=len(rf)
        )
    ]

100%|██████████| 4964/4964 [00:01<00:00, 4016.03it/s]


In [ ]:
rf["soil_clay"].describe()

count    4964.000000
mean      281.975423
std       100.451865
min         0.000000
25%       236.000000
50%       284.000000
75%       345.000000
max       529.000000
Name: soil_clay, dtype: float64

In [ ]:
with rasterio.open(
    "../data/raw/geo/india_soc_0_5cm.tif"
) as src:

    rf["soil_soc"] = [

        safe_sample(src, lon, lat)

        for lon, lat in tqdm(
            zip(rf.lon, rf.lat),
            total=len(rf)
        )
    ]

100%|██████████| 4964/4964 [00:01<00:00, 4144.90it/s]


In [ ]:
rf["soil_soc"].describe()

count    4473.000000
mean      225.736642
std       199.281159
min         0.000000
25%       103.000000
50%       167.000000
75%       270.000000
max      1231.000000
Name: soil_soc, dtype: float64

In [ ]:
from sklearn.neighbors import NearestNeighbors
coords = rf[
    ["lat","lon"]
].values

nn = NearestNeighbors(
    n_neighbors=5
)

nn.fit(coords)

dist, idx = nn.kneighbors(coords)
slopes = []

for i in range(len(rf)):

    neigh = idx[i]

    s = abs(

        rf.iloc[neigh]
          ["elevation"]
          .mean()

        -

        rf.iloc[i]
          ["elevation"]

    )

    slopes.append(s)

rf["slope"] = slopes

In [ ]:
rf.to_csv(
    "../data/processed/feature_matrix_geo_v1.csv",
    index=False
)

Water situation

In [ ]:
from pathlib import Path
import rasterio
from rasterio.merge import merge

tiles = list(
    Path("../data/raw/geo/jrc_water")
    .glob("*.tif")
)

print(len(tiles))

8


In [ ]:
from pathlib import Path
import rasterio
import pandas as pd
import numpy as np
df = pd.read_csv(
    "../data/processed/feature_matrix_geo_v1.csv"
)
water_tiles = list(
    Path("../data/raw/geo/jrc_water")
    .glob("*.tif")
)

print(len(water_tiles))

8


In [ ]:
def sample_water(lon, lat):

    for tile in water_tiles:

        try:

            with rasterio.open(tile) as src:

                left, bottom, right, top = src.bounds

                if (
                    left <= lon <= right
                    and
                    bottom <= lat <= top
                ):

                    val = next(
                        src.sample(
                            [(lon, lat)]
                        )
                    )[0]

                    return val

        except:
            pass

    return np.nan

In [ ]:
from tqdm import tqdm

df["water_occurrence"] = [

    sample_water(lon, lat)

    for lon, lat in tqdm(
        zip(df.lon, df.lat),
        total=len(df)
    )

]
df["water_occurrence"].describe()

100%|██████████| 4964/4964 [00:11<00:00, 423.13it/s]


count    3838.000000
mean        2.843669
std        15.597092
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       255.000000
Name: water_occurrence, dtype: float64

In [ ]:
df["water_occurrence"] = df["water_occurrence"].replace(
    255,
    np.nan
)


df["water_occurrence"].describe()

count    3837.000000
mean        2.777952
std        15.058323
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       100.000000
Name: water_occurrence, dtype: float64

In [ ]:
df["water_occurrence"] = (
    df["water_occurrence"]
    .fillna(0)
)

df["water_norm"] = (
    df["water_occurrence"] / 100
)

In [ ]:
print(df.columns.tolist())

df["water_occurrence"].describe()

['lat', 'lon', 'mean_annual_rf', 'std_annual', 'cv', 'dry_days', 'heavy_days', 'elevation', 'soil_clay', 'soil_soc', 'slope', 'water_occurrence', 'water_norm']


count    4964.000000
mean        2.147260
std        13.289712
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       100.000000
Name: water_occurrence, dtype: float64

In [ ]:
df.to_csv(
    "../data/processed/feature_matrix_geo_v2.csv",
    index=False
)